# Word retrieval, step by step

Goal: find the passages that answer a question.
Each level adds one idea. Run top to bottom.
Last level saves embeddings for the agent's `semantic_search` tool.

Example: a synthetic plan handbook, built from `development/fixtures/documents/`.
Made on first run, then uploaded to the document library.

## Setup

In [ ]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # find the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from backend.agents.embeddings import Embedder
from backend.storage.s3 import configured_store

store = configured_store()  # local library, or S3 when DOCUMENTS_STORAGE=s3
embedder = Embedder()       # Titan Text Embeddings v2
_cache = {}

def embed_all(texts):
    """One vector per text. Same text is embedded once."""
    new = [t for t in dict.fromkeys(texts) if t not in _cache]
    with ThreadPoolExecutor(max_workers=8) as pool:
        for text, vector in zip(new, pool.map(embedder.embed, new)):
            _cache[text] = vector
    return np.array([_cache[t] for t in texts])

def search(chunks, vectors, query, k=5):
    """Top k chunks by similarity. Vectors are unit length, so dot product = cosine."""
    scores = vectors @ embed_all([query])[0]
    best = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]

def show(results, chars=300):
    for score, chunk in results:
        print(f"--- {score:.3f}\n{chunk[:chars]}\n")

FILENAME = "bingle_dingle_handbook.docx"
QUERY = "How much does a member pay for a virtual visit?"

import re
from docx import Document as WordDocument
from backend.preprocessing.documents import ingest_document

def build_handbook():
    """Synthetic handbook: one heading per fixture, numbered sections as subheadings, rates as a table."""
    word = WordDocument()
    word.add_heading("Bingle-Dingle Insurance plan handbook", 0)
    for path in sorted((ROOT / "development/fixtures/documents").glob("bingle_dingle_*.txt")):
        lines = [l.strip() for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
        word.add_heading(lines[0], 1)
        rates = [l for l in lines if re.match(r"DEMO-[A-Z]+: .+; USD", l)]
        for line in lines[1:]:
            if re.match(r"^(\d+\.|Section [A-Z])\s", line):
                word.add_heading(line, 2)
            elif line in rates:
                if line == rates[0]:
                    table = word.add_table(rows=1, cols=4)
                    for cell, text in zip(table.rows[0].cells, ["Code", "Service", "Rate", "Limit"]):
                        cell.text = text
                    for rate in rates:
                        code, rest = rate.split(": ", 1)
                        for cell, text in zip(table.add_row().cells, [code, *rest.split("; ")]):
                            cell.text = text
            else:
                word.add_paragraph(line)
    path = ROOT / "data" / "examples" / FILENAME
    path.parent.mkdir(parents=True, exist_ok=True)
    word.save(path)
    return path

try:
    doc = store.load(FILENAME)
except Exception:
    doc = ingest_document(FILENAME, build_handbook().read_bytes(), store=store)
print(len(doc.blocks), "blocks. First three:")
for block in doc.blocks[:3]:
    print(block["location"], "|", block["text"][:80])

## Level 1: one paragraph = one chunk

Simplest. Preprocessing already split the file into blocks.
Paragraphs are blocks. Each table row is a block too (`cell | cell`).
Weakness: a short paragraph loses its context.

In [ ]:
chunks = [b["text"] for b in doc.blocks]
vectors = embed_all(chunks)
show(search(chunks, vectors, QUERY))

## Level 2: add the section heading

Each chunk starts with its heading.
"20 percent" alone is vague. Under "Benefit categories" it is clear.

In [ ]:
def with_section(block):
    section = block["location"].get("section", "")
    return f"[{section}]\n{block['text']}" if section else block["text"]

chunks = [with_section(b) for b in doc.blocks]
vectors = embed_all(chunks)
show(search(chunks, vectors, QUERY))
level2_chunks, level2_vectors = chunks, vectors  # reused by a later level

## Level 3: merge paragraphs up to a size

Join neighbors in the same section until `MAX_CHARS`.
More context per chunk. Fewer chunks.
Too big: one chunk covers many topics, matches blur.

In [ ]:
MAX_CHARS = 800

def merged(blocks, max_chars=MAX_CHARS):
    chunks, current, section = [], [], None
    for b in blocks:
        s = b["location"].get("section", "")
        if current and (s != section or sum(len(t) for t in current) + len(b["text"]) > max_chars):
            chunks.append(current)
            current = []
        current.append(b["text"])
        section = s
    if current:
        chunks.append(current)
    return chunks

groups = merged(doc.blocks)
chunks = ["\n".join(g) for g in groups]
vectors = embed_all(chunks)
print(len(chunks), "chunks, from", len(doc.blocks), "blocks")
show(search(chunks, vectors, QUERY))

## Level 4: overlap

Each chunk repeats the last paragraph of the one before.
An answer split across a boundary still lands in one chunk.
Cost: some text is embedded twice.

In [ ]:
OVERLAP = 1  # paragraphs carried into the next chunk

overlapped = [groups[0]] + [prev[-OVERLAP:] + g for prev, g in zip(groups, groups[1:])]
chunks = ["\n".join(g) for g in overlapped]
vectors = embed_all(chunks)
show(search(chunks, vectors, QUERY))

## Level 5: drop weak matches

Search always returns k results, even bad ones.
A minimum score drops the weak ones.
Pick it by looking at scores for good and bad queries.
Uses the level 2 chunks.

In [ ]:
MIN_SCORE = 0.35

def search_filtered(chunks, vectors, query, k=5, min_score=MIN_SCORE):
    return [(s, c) for s, c in search(chunks, vectors, query, k) if s >= min_score]

for q in [QUERY, "Who won the 1998 World Cup?"]:
    print("Query:", q)
    show(search_filtered(level2_chunks, level2_vectors, q), chars=120)

## Level 6: save for the agent

Uses the backend code the agent uses: one vector per block (level 1 chunks).
Saves `embeddings.json` in this file's `preprocessed/` folder.
The agent offers `semantic_search` when `SEMANTIC_SEARCH=true`.

In [ ]:
from backend.agents.embeddings import embed_document
from backend.retrieval.semantic import semantic_search_documents

embeddings = embed_document(doc, embedder)
print({k: v for k, v in embeddings["meta"].items() if k != "failed_blocks"})
store.save_embeddings(doc.storage_ref["manifest_key"], embeddings)

saved = store.load(FILENAME)  # read back what the agent will read
for m in semantic_search_documents({saved.id: saved}, QUERY, embedder, limit=5)["matches"]:
    print(f"--- {m['similarity']:.3f}  {m['location']}\n{m['text'][:300]}\n")